# Demands — scénario `reality_access`

Construit `Demands.csv` par cluster pour le scénario **reality_access** : l'accès universel à
l'électricité au niveau de service *sufficiency* pour les ménages qui n'ont pas le réseau.

| Composante | Provenance | Traitement |
|---|---|---|
| **A** — ménages/services raccordés au réseau | mesures AETN (`source_A_all_sectors_end_uses.csv`) | **inchangée** vs `reality` |
| **B** — ménages hors réseau (motor propio, panel solar, otra fuente) | RAMP **sufficiency**, au prorata | **remplace** le RAMP `reality` de B |
| **C** — ménages non électrifiés (no tiene) | RAMP **sufficiency**, au prorata | **nouvelle** demande |
| **Cooking** — énergie utile de cuisson | recensement 2024 | **inchangée** vs `reality` |

$$\text{Demands}_{\text{access}} = \text{Demands}_{\text{reality}} - B_{\text{reality}} + (B+C)_{\text{sufficiency}}$$

**Méthode B+C** — RAMP n'est **pas relancé**. Pour chaque municipalité, les sorties RAMP
sufficiency (qui couvrent l'ensemble des ménages) sont mises au prorata par

$$f = \frac{HH_B + HH_C}{HH_{\text{total}}}$$

lu depuis le recensement 2024 (`CSV_final.csv`), puis agrégées par cluster.

> **Note — `sufficiency_water_heating`.** Le `MAPPING` du notebook `sufficiency/demande.ipynb`
> omet cette colonne (71,7 GWh/an, 43 % de la demande ménages RAMP). Elle est reprise ici en
> `HOUSEHOLDS × HEAT_LOW_T_HW`. Sans elle, la demande B+C tomberait à ~1 129 kWh/HH au lieu des
> ~1 979 kWh/HH attendus par le breakeven GIS (`phase2_share_dispersion.ipynb`), qui somme bien
> toutes les colonnes `sufficiency_*`.

Sortie : `output_energyscope/C{k}/Demands.csv` — dossier distinct, les Demands `reality` et
`sufficiency` ne sont pas écrasés.

## 0. Contrôle — `Layers_in_out.csv` vs référence EnergyScope

Avant de construire `Demands.csv`, vérifie que le `Layers_in_out.csv` local (utilisé plus bas
pour convertir l'énergie électrique RAMP/AETN en énergie utile) correspond au fichier de
référence utilisé par le modèle EnergyScope lui-même, dans
`EnergyScope_BO_nord_amazonia/Data/2025/reality_access/00_INDEP/`. Une divergence silencieuse
ici fausserait tous les coefficients d'efficacité calculés en aval.

In [1]:
import pandas as pd

LIO_LOCAL_PATH = "../data/Layers_in_out.csv"
LIO_REFERENCE_PATH = "../../../EnergyScope_BO_nord_amazonia/Data/2025/reality_access/00_INDEP/Layers_in_out.csv"

lio_local = pd.read_csv(LIO_LOCAL_PATH, sep=";", header=0, index_col=0)
lio_reference = pd.read_csv(LIO_REFERENCE_PATH, sep=";", header=0, index_col=0)

problems = []

only_local_rows = sorted(set(lio_local.index) - set(lio_reference.index))
only_reference_rows = sorted(set(lio_reference.index) - set(lio_local.index))
if only_local_rows:
    problems.append(f"technologies only in local file: {only_local_rows}")
if only_reference_rows:
    problems.append(f"technologies only in EnergyScope reference: {only_reference_rows}")

only_local_cols = sorted(set(lio_local.columns) - set(lio_reference.columns))
only_reference_cols = sorted(set(lio_reference.columns) - set(lio_local.columns))
if only_local_cols:
    problems.append(f"layers only in local file: {only_local_cols}")
if only_reference_cols:
    problems.append(f"layers only in EnergyScope reference: {only_reference_cols}")

common_rows = sorted(set(lio_local.index) & set(lio_reference.index))
common_cols = sorted(set(lio_local.columns) & set(lio_reference.columns))
changed_rows = sorted(
    tech for tech in common_rows
    if not lio_local.loc[tech, common_cols].equals(lio_reference.loc[tech, common_cols])
)
if changed_rows:
    problems.append(f"technologies with different coefficients: {changed_rows}")

if problems:
    raise ValueError(
        f"Layers_in_out.csv differs from the EnergyScope reference ({LIO_REFERENCE_PATH}): "
        + "; ".join(problems)
    )

print(f"OK — Layers_in_out.csv matches the EnergyScope reference ({LIO_REFERENCE_PATH})")

OK — Layers_in_out.csv matches the EnergyScope reference (../../../EnergyScope_BO_nord_amazonia/Data/2025/reality_access/00_INDEP/Layers_in_out.csv)


## 1. Configuration — clusters, correspondances de noms, mappings RAMP

In [2]:
import os
import pandas as pd

OUTPUT_DIR = "output_energyscope"

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}
MUNI_TO_CLUSTER = {m: k for k, munis in CLUSTERS.items() for m in munis}

SECTOR_COLS = ["HOUSEHOLDS", "SERVICES", "INDUSTRY", "TRANSPORTATION",
               "PUBLIC_LIGHTING", "AGRICULTURE", "MINING", "FISHING_OTHERS"]

# Source A uses display names with spaces; canonical names use underscores
SOURCE_A_TO_RAMP = {
    "Bella Flor": "Bella_Flor", "Bolpebra": "Bolpebra", "Cobija": "Cobija",
    "El Sena": "Sena", "Exaltación": "Exaltación", "Filadelfia": "Filadelfia",
    "Guayaramerín": "Guayaramerín", "Ingavi": "Ingavi", "Ixiamas": "Ixiamas",
    "Nueva Esperanza": "Nueva_Esperanza", "Porvenir": "Porvenir",
    "Puerto Gonzalo Moreno": "Puerto_Gonzalo_Moreno", "Puerto Rico": "Puerto_Rico",
    "Reyes": "Reyes", "Riberalta": "Riberalta", "San Lorenzo": "San_Lorenzo",
    "San Pedro": "San_Pedro", "Santa Rosa": "Santa_Rosa_Beni",
    "Santa Rosa del Abuná": "Santa_Rosa_Pando", "Santos Mercado": "Santos_Mercado",
    "Villa Nueva": "Villa_Nueva",
}

# Census has two "Santa Rosa" — disambiguated by (name, department)
CSVFINAL_TO_RAMP = {
    ("Ixiamas", "La Paz"): "Ixiamas", ("Riberalta", "Beni"): "Riberalta",
    ("Guayaramerín", "Beni"): "Guayaramerín", ("Reyes", "Beni"): "Reyes",
    ("Santa Rosa", "Beni"): "Santa_Rosa_Beni", ("Exaltación", "Beni"): "Exaltación",
    ("Cobija", "Pando"): "Cobija", ("Porvenir", "Pando"): "Porvenir",
    ("Bolpebra", "Pando"): "Bolpebra", ("Bella Flor", "Pando"): "Bella_Flor",
    ("Puerto Rico", "Pando"): "Puerto_Rico", ("San Pedro", "Pando"): "San_Pedro",
    ("Filadelfia", "Pando"): "Filadelfia",
    ("Puerto Gonzalo Moreno", "Pando"): "Puerto_Gonzalo_Moreno",
    ("San Lorenzo", "Pando"): "San_Lorenzo", ("Sena", "Pando"): "Sena",
    ("Santa Rosa", "Pando"): "Santa_Rosa_Pando", ("Ingavi", "Pando"): "Ingavi",
    ("Nueva Esperanza", "Pando"): "Nueva_Esperanza", ("Villa Nueva", "Pando"): "Villa_Nueva",
    ("Santos Mercado", "Pando"): "Santos_Mercado",
}

In [3]:
# RAMP sufficiency column → (EnergyScope sector, end-use layer).
# Identical to sufficiency/demande.ipynb EXCEPT sufficiency_water_heating, which that
# notebook omits — see the note at the top.
MAPPING_SUFF = {
    "sufficiency_illumination":            ("HOUSEHOLDS", "LIGHTING_R_C"),
    "sufficiency_ICT":                     ("HOUSEHOLDS", "ELECTRICITY"),
    "sufficiency_cold_storage":            ("HOUSEHOLDS", "FOOD_PRESERVATION"),
    "sufficiency_thermal_comfort":         ("HOUSEHOLDS", "SPACE_COOLING"),
    "sufficiency_water_heating":           ("HOUSEHOLDS", "HEAT_LOW_T_HW"),
    "big_school_illumination":             ("SERVICES",   "LIGHTING_R_C"),
    "big_school_ICT":                      ("SERVICES",   "ELECTRICITY"),
    "big_school_cold_storage":             ("SERVICES",   "FOOD_PRESERVATION"),
    "big_school_space_cooling":            ("SERVICES",   "SPACE_COOLING"),
    "health_center_illumination":          ("SERVICES",   "LIGHTING_R_C"),
    "health_center_ICT":                   ("SERVICES",   "ELECTRICITY"),
    "health_center_cold_storage":          ("SERVICES",   "FOOD_PRESERVATION"),
    "health_center_space_cooling":         ("SERVICES",   "SPACE_COOLING"),
    "health_center_water_heating":         ("SERVICES",   "HEAT_LOW_T_HW"),
    "health_center_water_supply":          ("SERVICES",   "ELECTRICITY"),
    "health_center_medical_equip":         ("SERVICES",   "ELECTRICITY"),
    "entertainment_business_illumination": ("SERVICES",   "LIGHTING_R_C"),
    "entertainment_business_ICT":          ("SERVICES",   "ELECTRICITY"),
    "entertainment_business_cold_storage": ("SERVICES",   "FOOD_PRESERVATION"),
    "restaurant_illumination":             ("SERVICES",   "LIGHTING_R_C"),
    "restaurant_cold_storage":             ("SERVICES",   "FOOD_PRESERVATION"),
    "restaurant_kitchen":                  ("SERVICES",   "COOKING"),
    "store_illumination":                  ("SERVICES",   "LIGHTING_R_C"),
    "store_ICT":                           ("SERVICES",   "ELECTRICITY"),
    "store_cold_storage":                  ("SERVICES",   "FOOD_PRESERVATION"),
    "workshop_illumination":               ("SERVICES",   "LIGHTING_R_C"),
    "workshop_ICT":                        ("SERVICES",   "ELECTRICITY"),
    "workshop_machinery":                  ("SERVICES",   "MECHANICAL_ENERGY_COMM"),
    "public_lighting_illumination":        ("PUBLIC_LIGHTING", "LIGHTING_P"),
    "rice_processing_rice_processing":     ("INDUSTRY",   "MECHANICAL_ENERGY_IND"),
}

# RAMP reality mapping — needed only to subtract Source B reality (verification 2)
MAPPING_RAMP_REALITY = {
    "sufficiency_illumination":            ("HOUSEHOLDS", "LIGHTING_R_C"),
    "sufficiency_ICT":                     ("HOUSEHOLDS", "ELECTRICITY"),
    "sufficiency_cold_storage":            ("HOUSEHOLDS", "FOOD_PRESERVATION"),
    "sufficiency_thermal_comfort":         ("HOUSEHOLDS", "SPACE_COOLING"),
    "small_school_illumination":           ("SERVICES",   "LIGHTING_R_C"),
    "small_school_ICT":                    ("SERVICES",   "ELECTRICITY"),
    "entertainment_business_illumination": ("SERVICES",   "LIGHTING_R_C"),
    "entertainment_business_ICT":          ("SERVICES",   "ELECTRICITY"),
    "entertainment_business_cold_storage": ("SERVICES",   "FOOD_PRESERVATION"),
    "rice_processing_rice_processing":     ("INDUSTRY",   "MECHANICAL_ENERGY_IND"),
    "restaurant_illumination":             ("SERVICES",   "LIGHTING_R_C"),
    "restaurant_cold_storage":             ("SERVICES",   "FOOD_PRESERVATION"),
    "restaurant_kitchen":                  ("SERVICES",   "COOKING"),
    "store_illumination":                  ("SERVICES",   "LIGHTING_R_C"),
    "store_ICT":                           ("SERVICES",   "ELECTRICITY"),
    "store_cold_storage":                  ("SERVICES",   "FOOD_PRESERVATION"),
    "workshop_illumination":               ("SERVICES",   "LIGHTING_R_C"),
    "workshop_ICT":                        ("SERVICES",   "ELECTRICITY"),
    "workshop_machinery":                  ("SERVICES",   "MECHANICAL_ENERGY_COMM"),
}

## 2. Coefficients d'efficacité (`Layers_in_out.csv`)

RAMP et Source A sont en **électricité finale** ; EnergyScope attend l'**énergie utile de service**.

$$\text{GWh}_{\text{utile}} = \frac{\text{GWh}_{\text{électrique}}}{|\text{coeff}_{\text{ELECTRICITY}}|}$$

Mêmes coefficients que les notebooks `reality` et `sufficiency`.

In [4]:
LAYER_TO_TECH = {
    'ELECTRICITY':                   None,
    'LIGHTING_R_C':                  'LED_BULB',
    'LIGHTING_P':                    'LED_LIGHT',
    'HEAT_HIGH_T':                   'IND_DIRECT_ELEC',
    'HEAT_LOW_T_SH':                 'DEC_DIRECT_ELEC',
    'HEAT_LOW_T_HW':                 'DEC_DIRECT_ELEC',
    'COOKING':                       'STOVE_ELEC',
    'PROCESS_COOLING':               'IND_ELEC_COLD',
    'SPACE_COOLING':                 'DEC_ELEC_COLD',
    'FOOD_PRESERVATION':             'REFRIGERATOR_EL',
    'MECHANICAL_ENERGY_COMM':        'COMM_MACHINERY_EL',
    'MECHANICAL_ENERGY_IND':         'IND_MACHINERY_EL',
    'MECHANICAL_ENERGY_MOV_AGR':     'TRACTOR_EL',
    'MECHANICAL_ENERGY_FIX_AGR':     'AGR_MACHINERY_EL',
    'MECHANICAL_ENERGY_MIN':         'MIN_MACHINERY_EL',
    'MECHANICAL_ENERGY_FISH_OTHERS': 'FISH_MACHINERY_EL',
    'NON_ENERGY':                    None,
}
MOBILITY_LAYERS = {'MOBILITY_PASSENGER', 'MOBILITY_FREIGHT', 'AVIATION_LONG_HAUL', 'SHIPPING'}

lio = pd.read_csv("../data/Layers_in_out.csv", sep=";", header=0, index_col=0)

LAYER_TO_COEFF = {
    layer: (1.0 if tech is None else abs(float(lio.loc[tech, "ELECTRICITY"])))
    for layer, tech in LAYER_TO_TECH.items()
}

def to_useful(gwh_elec, end_use):
    return gwh_elec / LAYER_TO_COEFF.get(end_use, 1.0)

print("Layer → |ELECTRICITY coefficient|:")
for layer, coeff in LAYER_TO_COEFF.items():
    print(f"  {layer:<35} {coeff:.6f}")

Layer → |ELECTRICITY coefficient|:
  ELECTRICITY                         1.000000
  LIGHTING_R_C                        2.941176
  LIGHTING_P                          2.941176
  HEAT_HIGH_T                         1.000000
  HEAT_LOW_T_SH                       1.000000
  HEAT_LOW_T_HW                       1.000000
  COOKING                             1.000000
  PROCESS_COOLING                     0.496500
  SPACE_COOLING                       0.400000
  FOOD_PRESERVATION                   2.792308
  MECHANICAL_ENERGY_COMM              1.212121
  MECHANICAL_ENERGY_IND               1.111111
  MECHANICAL_ENERGY_MOV_AGR           1.111111
  MECHANICAL_ENERGY_FIX_AGR           1.111111
  MECHANICAL_ENERGY_MIN               1.111111
  MECHANICAL_ENERGY_FISH_OTHERS       1.111111
  NON_ENERGY                          1.000000


## 3. Table `Demands` vide

21 lignes (une par couche d'usage final), tous secteurs à zéro. Format identique aux
`Demands.csv` de `reality` et `sufficiency`.

In [5]:
def create_empty_demands():
    columns = ["Category", "Subcategory", "parameter name"] + SECTOR_COLS + ["Units"]
    rows = [
        ["Electricity", "Electricity",                            "ELECTRICITY",                   "[GWh]"],
        ["Lighting",    "Building lighting",                      "LIGHTING_R_C",                  "[GWh]"],
        ["Lighting",    "Public lighting",                        "LIGHTING_P",                    "[GWh]"],
        ["Heat",        "High temperature",                       "HEAT_HIGH_T",                   "[GWh]"],
        ["Heat",        "Space heating",                          "HEAT_LOW_T_SH",                 "[GWh]"],
        ["Heat",        "Hot water",                              "HEAT_LOW_T_HW",                 "[GWh]"],
        ["Heat",        "Cooking",                                "COOKING",                       "[GWh]"],
        ["Cold",        "Process cooling",                        "PROCESS_COOLING",               "[GWh]"],
        ["Cold",        "Space cooling",                          "SPACE_COOLING",                 "[GWh]"],
        ["Cold",        "Food preservation",                      "FOOD_PRESERVATION",             "[GWh]"],
        ["Mobility",    "Passenger",                              "MOBILITY_PASSENGER",            "[Mpkm]"],
        ["Mobility",    "Freight",                                "MOBILITY_FREIGHT",              "[Mtkm]"],
        ["Mobility",    "Long-haul passenger flights",            "AVIATION_LONG_HAUL",            "[Mpkm]"],
        ["Mobility",    "International shipping",                 "SHIPPING",                      "[Mtkm]"],
        ["Mechanical",  "Mechanical energy commercial",           "MECHANICAL_ENERGY_COMM",        "[GWh]"],
        ["Mechanical",  "Mechanical energy industrial",           "MECHANICAL_ENERGY_IND",         "[GWh]"],
        ["Mechanical",  "Mechanical energy agriculture mobility", "MECHANICAL_ENERGY_MOV_AGR",     "[GWh]"],
        ["Mechanical",  "Mechanical energy agriculture fixed",    "MECHANICAL_ENERGY_FIX_AGR",     "[GWh]"],
        ["Mechanical",  "Mechanical energy mining",               "MECHANICAL_ENERGY_MIN",         "[GWh]"],
        ["Mechanical",  "Mechanical energy fishing",              "MECHANICAL_ENERGY_FISH_OTHERS", "[GWh]"],
        ["Non-energy",  "Non-energy",                             "NON_ENERGY",                    "[GWh]"],
    ]
    df = pd.DataFrame(columns=columns)
    for i, row in enumerate(rows):
        df.loc[i] = [row[0], row[1], row[2]] + [0.0] * 8 + [row[3]]
    return df


def add_demands(df, contributions):
    """Add {(sector, end_use): GWh} into a Demands table, in place."""
    for (sector, end_use), gwh in contributions.items():
        if end_use in MOBILITY_LAYERS or sector not in SECTOR_COLS:
            continue
        df.loc[df["parameter name"] == end_use, sector] += gwh
    return df

## 4. Ménages par source d'électricité — recensement 2024

Aucun effectif n'est codé en dur : tout est lu depuis `CSV_final.csv`, section
*NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024*.

- **A** = `Servicio público de energía eléctrica` (réseau)
- **B** = `Motor propio` + `Panel solar` + `Otra` — hors réseau, auto-approvisionnés
- **C** = `No tiene` — non électrifiés

$$f_{\text{muni}} = \frac{HH_B + HH_C}{HH_{\text{total}}}$$

In [6]:
COL_DEPT, COL_MUNI = 1, 3
# 2024 electricity-source columns
COL_HH_TOTAL, COL_HH_PUBLIC, COL_HH_MOTOR, COL_HH_SOLAR, COL_HH_OTRA, COL_HH_NONE = 14, 15, 16, 17, 18, 19

# header=None keeps integer column positions aligned with the original Excel layout
csv_final = pd.read_csv("../../exctraction of data/output/CSV_final.csv", header=None)


def parse_int_cell(x):
    if pd.isna(x):
        return 0
    s = str(x).strip().replace("\xa0", "").replace(" ", "")
    if s in ("", "-", "nan"):
        return 0
    try:
        return int(float(s))
    except ValueError:
        return 0


records = []
for i in range(1, csv_final.shape[0]):  # row 0 is the header; province rows fail the key lookup
    key = (str(csv_final.iloc[i, COL_MUNI]).strip(), str(csv_final.iloc[i, COL_DEPT]).strip())
    if key not in CSVFINAL_TO_RAMP:
        continue
    muni = CSVFINAL_TO_RAMP[key]
    motor = parse_int_cell(csv_final.iloc[i, COL_HH_MOTOR])
    solar = parse_int_cell(csv_final.iloc[i, COL_HH_SOLAR])
    otra  = parse_int_cell(csv_final.iloc[i, COL_HH_OTRA])
    records.append({
        "Municipio":  muni,
        "Cluster":    MUNI_TO_CLUSTER[muni],
        "HH_total":   parse_int_cell(csv_final.iloc[i, COL_HH_TOTAL]),
        "HH_A_red":   parse_int_cell(csv_final.iloc[i, COL_HH_PUBLIC]),
        "Motor":      motor,
        "Panel":      solar,
        "Otra":       otra,
        "HH_B":       motor + solar + otra,
        "HH_C":       parse_int_cell(csv_final.iloc[i, COL_HH_NONE]),
    })

hh = pd.DataFrame(records).sort_values(["Cluster", "Municipio"]).reset_index(drop=True)
hh["HH_BC"]  = hh["HH_B"] + hh["HH_C"]
hh["factor"] = hh["HH_BC"] / hh["HH_total"]

assert set(hh["Municipio"]) == set(MUNI_TO_CLUSTER), "missing municipalities in census"
# A + B + C must reconstruct the census total exactly
assert (hh["HH_A_red"] + hh["HH_B"] + hh["HH_C"] == hh["HH_total"]).all(), "A+B+C != total"

print("Ménages par source d'électricité — recensement 2024")
print(hh.to_string(index=False, formatters={"factor": "{:.4f}".format}))
print()
print(f"RÉGION : HH_total={hh.HH_total.sum():,}  A_réseau={hh.HH_A_red.sum():,}  "
      f"B_hors_réseau={hh.HH_B.sum():,}  C_non_électrifiés={hh.HH_C.sum():,}  "
      f"B+C={hh.HH_BC.sum():,}  ({100*hh.HH_BC.sum()/hh.HH_total.sum():.1f} % des ménages)")

FACTOR = dict(zip(hh["Municipio"], hh["factor"]))
HH_BC_BY_CLUSTER = hh.groupby("Cluster")["HH_BC"].sum().to_dict()

Ménages par source d'électricité — recensement 2024
            Municipio  Cluster  HH_total  HH_A_red  Motor  Panel  Otra  HH_B  HH_C  HH_BC factor
           Exaltación        1      1455       292    203    664    23   890   273   1163 0.7993
              Ixiamas        1      3306      1470    337    353   103   793  1043   1836 0.5554
                Reyes        1      3417      2353     96    213    44   353   711   1064 0.3114
      Santa_Rosa_Beni        1      2755      1728    198    405    17   620   407   1027 0.3728
             Bolpebra        2       802       156    160    216    17   393   253    646 0.8055
         Guayaramerín        3     10891      9420    166    383    97   646   825   1471 0.1351
Puerto_Gonzalo_Moreno        3      1965      1385     48     58   126   232   348    580 0.2952
            Riberalta        3     27442     23121    422    692   282  1396  2925   4321 0.1575
           Bella_Flor        4      1235       569     67    160    88   31

## 5. Source A — demande réseau (mesures AETN)

Reprise **telle quelle** du notebook `reality/demande.ipynb` : mêmes fichier, mêmes
regroupements, même conversion MWh final → GWh utile. `SERVICES_OTHER` est fusionné dans
`SERVICES`.

In [7]:
sa = pd.read_csv("../../exctraction of data/output/source_A_all_sectors_end_uses.csv")
sa["muni_ramp"] = sa["municipality"].map(SOURCE_A_TO_RAMP)

unmapped = sa[sa["muni_ramp"].isna()]["municipality"].unique()
if len(unmapped) > 0:
    raise ValueError(f"Unmapped Source A municipalities: {unmapped}")

sa["sector"]  = sa["sector"].replace("SERVICES_OTHER", "SERVICES")
sa["cluster"] = sa["muni_ramp"].map(MUNI_TO_CLUSTER)

source_a_by_cluster = {}
print("Source A — demande réseau par cluster (MWh final → GWh utile) :")
for cluster_id in sorted(CLUSTERS):
    sa_c = sa[sa["cluster"] == cluster_id]
    result = {
        (sector, end_use): to_useful(group["MWh"].sum() / 1000.0, end_use)
        for (sector, end_use), group in sa_c.groupby(["sector", "end_use"])
    }
    source_a_by_cluster[cluster_id] = result
    print(f"  C{cluster_id}: {sum(result.values()):8.4f} GWh utile  "
          f"(depuis {sa_c['MWh'].sum()/1000:8.4f} GWh final)")

Source A — demande réseau par cluster (MWh final → GWh utile) :
  C1:  11.4018 GWh utile  (depuis  11.7483 GWh final)
  C2:   0.2721 GWh utile  (depuis   0.2427 GWh final)
  C3:  90.6238 GWh utile  (depuis  90.0889 GWh final)
  C4:  23.6977 GWh utile  (depuis  22.8816 GWh final)
  C5:  55.3653 GWh utile  (depuis  55.1415 GWh final)


## 6. Cooking — recensement 2024, inchangé vs `reality`

Ménages comptés : `total_2024 − no_cocina_2024 − electricidad_2024`. Les cuisinières
électriques sont exclues : leur consommation est déjà dans Source A.

Intensité : 1 344 kWh/ménage/an d'énergie **utile**, indépendante du combustible
(estimation Pablo Jimenez Zabalaga, thèse Roger Arias 2024–2025). EnergyScope choisit
ensuite le mix STOVE_WOOD / STOVE_LPG / STOVE_ELEC.

Le scénario `reality_access` ne change **pas** la cuisson : l'accès à l'électricité ne
modifie pas le besoin utile de cuisson, seulement le mix que l'optimiseur peut choisir.

In [8]:
USEFUL_COOKING_PER_HH_GWh = 0.001344023  # GWh/household/year

COL_COOK_TOTAL, COL_COOK_ELEC, COL_NO_COCINA = 46, 52, 54  # 2024 cooking section

cooking_hh = {}
for i in range(1, csv_final.shape[0]):
    key = (str(csv_final.iloc[i, COL_MUNI]).strip(), str(csv_final.iloc[i, COL_DEPT]).strip())
    if key not in CSVFINAL_TO_RAMP:
        continue
    total     = parse_int_cell(csv_final.iloc[i, COL_COOK_TOTAL])
    elec      = parse_int_cell(csv_final.iloc[i, COL_COOK_ELEC])
    no_cocina = parse_int_cell(csv_final.iloc[i, COL_NO_COCINA])
    cooking_hh[CSVFINAL_TO_RAMP[key]] = total - no_cocina - elec

total_cooking_hh = sum(cooking_hh.values())
print(f"Ménages cuisson non électrique : {total_cooking_hh:,} "
      f"(attendu 82 328 = 82 501 − 173 électriques)")
assert total_cooking_hh == 82328, f"Mismatch: got {total_cooking_hh}"

cooking_by_cluster = {}
for cluster_id in sorted(CLUSTERS):
    n   = sum(cooking_hh[m] for m in CLUSTERS[cluster_id])
    gwh = n * USEFUL_COOKING_PER_HH_GWh
    cooking_by_cluster[cluster_id] = gwh
    print(f"  C{cluster_id}: {n:>6,} HH → {gwh:7.4f} GWh utile")

Ménages cuisson non électrique : 82,328 (attendu 82 328 = 82 501 − 173 électriques)
  C1: 10,698 HH → 14.3784 GWh utile
  C2:    794 HH →  1.0672 GWh utile
  C3: 39,489 HH → 53.0741 GWh utile
  C4: 16,343 HH → 21.9654 GWh utile
  C5: 15,004 HH → 20.1657 GWh utile


## 7. Demande B+C — prorata des sorties RAMP sufficiency

**RAMP n'est pas relancé.** Pour chaque municipalité on lit la courbe de charge sufficiency
annuelle (W, pas de 1 minute, année entière), on multiplie **chaque catégorie d'usage final**
par $f_{\text{muni}}$, puis on convertit en GWh utile.

$$\text{GWh}_{\text{élec}} = \frac{\sum_t P_t \,[\mathrm{W}] \times 1\,\mathrm{min}}{60 \times 10^{9}} \times f_{\text{muni}}$$

In [9]:
SUFF_RAMP_DIR = "../sufficiency/data ramp"

bc_by_cluster      = {}   # cluster → {(sector, end_use): GWh useful}
bc_hh_kwh_elec     = {}   # cluster → household-sector electric kWh (for the 1 979 check)
bc_muni_rows       = []

for cluster_id in sorted(CLUSTERS):
    contributions = {}
    hh_elec_gwh   = 0.0
    for muni in CLUSTERS[cluster_id]:
        path = f"{SUFF_RAMP_DIR}/{muni}/load_curve_energy_service_full_year_Norte_Amazonia.csv"
        if not os.path.exists(path):
            raise FileNotFoundError(f"RAMP sufficiency output missing for '{muni}': {path}")
        df = pd.read_csv(path)

        unmapped = [c for c in df.columns if c != "time" and c not in MAPPING_SUFF]
        if unmapped:
            raise ValueError(f"{muni}: unmapped RAMP columns would be silently dropped: {unmapped}")

        f = FACTOR[muni]
        muni_hh_elec = 0.0
        for col, (sector, end_use) in MAPPING_SUFF.items():
            if col not in df.columns:
                continue
            gwh_elec = df[col].fillna(0).sum() / 60_000_000_000.0 * f
            contributions[(sector, end_use)] = contributions.get((sector, end_use), 0.0) + to_useful(gwh_elec, end_use)
            if sector == "HOUSEHOLDS":
                muni_hh_elec += gwh_elec
        hh_elec_gwh += muni_hh_elec

        n_bc = int(hh.loc[hh["Municipio"] == muni, "HH_BC"].iloc[0])
        bc_muni_rows.append({"Cluster": cluster_id, "Municipio": muni, "factor": f,
                             "HH_BC": n_bc, "HH_kWh_par_HH": muni_hh_elec * 1e6 / n_bc})

    bc_by_cluster[cluster_id]  = contributions
    bc_hh_kwh_elec[cluster_id] = hh_elec_gwh * 1e6

print("Prorata B+C par municipalité :")
print(pd.DataFrame(bc_muni_rows).to_string(
    index=False, formatters={"factor": "{:.4f}".format, "HH_kWh_par_HH": "{:.1f}".format}))

Prorata B+C par municipalité :
 Cluster             Municipio factor  HH_BC HH_kWh_par_HH
       1            Exaltación 0.7993   1163        1983.6
       1                 Reyes 0.3114   1064        1967.9
       1       Santa_Rosa_Beni 0.3728   1027        1986.4
       1               Ixiamas 0.5554   1836        1941.2
       2              Bolpebra 0.8055    646        1976.7
       3          Guayaramerín 0.1351   1471        1985.2
       3             Riberalta 0.1575   4321        1984.9
       3 Puerto_Gonzalo_Moreno 0.2952    580        1982.8
       4            Bella_Flor 0.5393    666        1979.0
       4            Filadelfia 0.4781   1202        1974.0
       4                Ingavi 0.8259    536        1982.1
       4       Nueva_Esperanza 0.8548    418        1977.8
       4              Porvenir 0.1928    430        1974.6
       4           Puerto_Rico 0.3243    645        1981.0
       4           San_Lorenzo 0.3038    562        1982.3
       4             San_

## 8. Vérification 1 — demande B+C par cluster

L'intensité ménages est comparée aux **1 979 kWh/HH** du breakeven GIS
(`analyse_GIS_phase2/phase2_share_dispersion.ipynb`, moyenne des `per_hh_kWh` sufficiency).

La comparaison porte sur l'électricité **finale** du secteur HOUSEHOLDS — c'est la grandeur
que calcule le notebook GIS (somme des colonnes `sufficiency_*`, sans conversion en utile).

In [10]:
GIS_BREAKEVEN_KWH_PER_HH = 1979.0   # analyse_GIS_phase2/phase2_share_dispersion.ipynb

rows = []
for cluster_id in sorted(CLUSTERS):
    n_bc     = HH_BC_BY_CLUSTER[cluster_id]
    kwh_hh   = bc_hh_kwh_elec[cluster_id] / n_bc
    rows.append({
        "Cluster":        f"C{cluster_id}",
        "HH_BC":          n_bc,
        "B+C_GWh_utile":  sum(bc_by_cluster[cluster_id].values()),
        "HH_GWh_élec":    bc_hh_kwh_elec[cluster_id] / 1e6,
        "kWh_par_HH":     kwh_hh,
        "écart_vs_1979_%": 100 * (kwh_hh / GIS_BREAKEVEN_KWH_PER_HH - 1),
    })
check1 = pd.DataFrame(rows)

tot_hh   = sum(HH_BC_BY_CLUSTER.values())
tot_kwh  = sum(bc_hh_kwh_elec.values()) / tot_hh
check1.loc[len(check1)] = {
    "Cluster": "RÉGION", "HH_BC": tot_hh,
    "B+C_GWh_utile": check1["B+C_GWh_utile"].sum(),
    "HH_GWh_élec":   check1["HH_GWh_élec"].sum(),
    "kWh_par_HH":    tot_kwh,
    "écart_vs_1979_%": 100 * (tot_kwh / GIS_BREAKEVEN_KWH_PER_HH - 1),
}

print("VÉRIFICATION 1 — demande B+C par cluster")
print(check1.to_string(index=False, formatters={
    "B+C_GWh_utile": "{:8.4f}".format, "HH_GWh_élec": "{:8.4f}".format,
    "kWh_par_HH": "{:7.1f}".format, "écart_vs_1979_%": "{:+.2f}".format}))

dev = abs(tot_kwh / GIS_BREAKEVEN_KWH_PER_HH - 1)
print()
print(f"→ Écart régional vs breakeven GIS : {100*dev:+.2f} %  "
      f"({'OK' if dev < 0.02 else 'HORS TOLÉRANCE'}, tolérance 2 %)")
assert dev < 0.02, "B+C household intensity inconsistent with the GIS breakeven"

VÉRIFICATION 1 — demande B+C par cluster
Cluster  HH_BC B+C_GWh_utile HH_GWh_élec kWh_par_HH écart_vs_1979_%
     C1   5090        7.9271     10.0048     1965.6           -0.68
     C2    646        1.0269      1.2770     1976.7           -0.11
     C3   6372       10.2395     12.6470     1984.8           +0.29
     C4   7963       12.7300     15.7755     1981.1           +0.11
     C5    643        1.0182      1.2708     1976.4           -0.13
 RÉGION  20714       32.9417     40.9750     1978.1           -0.04

→ Écart régional vs breakeven GIS : +0.04 %  (OK, tolérance 2 %)


## 9. Source B `reality` — à retirer

Le RAMP `reality` des 9 325 ménages hors réseau (B) décrit leur consommation **actuelle**
(~183 kWh/HH). Sous `reality_access` ces ménages passent au niveau sufficiency (~1 979 kWh/HH),
donc leur demande `reality` est **remplacée**, pas cumulée — sans quoi elle serait comptée deux fois.

In [11]:
ramp_reality = pd.read_csv("../reality/data ramp/ramp_reality_annual_summary.csv")
ramp_reality = ramp_reality[ramp_reality["municipality"] != "TOTAL"].copy()
ramp_reality["cluster"] = ramp_reality["municipality"].map(MUNI_TO_CLUSTER)

source_b_reality_by_cluster = {}
print("Source B reality — demande hors réseau à retirer (GWh utile) :")
for cluster_id in sorted(CLUSTERS):
    result = {}
    for _, row in ramp_reality[ramp_reality["cluster"] == cluster_id].iterrows():
        for col, (sector, end_use) in MAPPING_RAMP_REALITY.items():
            if col in row.index and not pd.isna(row[col]):
                key = (sector, end_use)
                result[key] = result.get(key, 0.0) + to_useful(float(row[col]), end_use)
    source_b_reality_by_cluster[cluster_id] = result
    print(f"  C{cluster_id}: {sum(result.values()):7.4f} GWh utile")

Source B reality — demande hors réseau à retirer (GWh utile) :
  C1:  0.5493 GWh utile
  C2:  0.0734 GWh utile
  C3:  0.6165 GWh utile
  C4:  0.8651 GWh utile
  C5:  0.0927 GWh utile


## 10. Construction des `Demands.csv` `reality_access`

Par cluster : Source A (réseau, inchangée) + cooking (inchangé) + B+C sufficiency (prorata).

In [12]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
demands_access = {}

for cluster_id in sorted(CLUSTERS):
    df = create_empty_demands()
    add_demands(df, source_a_by_cluster.get(cluster_id, {}))                       # A — réseau
    df.loc[df["parameter name"] == "COOKING", "HOUSEHOLDS"] += cooking_by_cluster[cluster_id]
    add_demands(df, bc_by_cluster[cluster_id])                                     # B+C — sufficiency

    out_path = f"{OUTPUT_DIR}/C{cluster_id}/Demands.csv"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.to_csv(out_path, sep=";", index=False)
    demands_access[cluster_id] = df

    print(f"Cluster {cluster_id} ({len(CLUSTERS[cluster_id])} municipalités) → {out_path}")
    total = 0.0
    for sec in SECTOR_COLS:
        val = df[sec].astype(float).sum()
        if val > 0:
            print(f"  {sec:<18}: {val:8.4f} GWh")
            total += val
    print(f"  {'TOTAL':<18}: {total:8.4f} GWh")
    print("-" * 46)

Cluster 1 (4 municipalités) → output_energyscope/C1/Demands.csv
  HOUSEHOLDS        :  29.3732 GWh
  SERVICES          :   3.2378 GWh
  INDUSTRY          :   0.7382 GWh
  PUBLIC_LIGHTING   :   0.3581 GWh
  TOTAL             :  33.7072 GWh
----------------------------------------------


Cluster 2 (1 municipalités) → output_energyscope/C2/Demands.csv
  HOUSEHOLDS        :   2.1922 GWh
  SERVICES          :   0.1434 GWh
  INDUSTRY          :   0.0256 GWh
  PUBLIC_LIGHTING   :   0.0050 GWh
  TOTAL             :   2.3661 GWh
----------------------------------------------


Cluster 3 (3 municipalités) → output_energyscope/C3/Demands.csv
  HOUSEHOLDS        : 115.2303 GWh
  SERVICES          :  30.2907 GWh
  INDUSTRY          :   6.5795 GWh
  PUBLIC_LIGHTING   :   1.8370 GWh
  TOTAL             : 153.9374 GWh
----------------------------------------------
Cluster 4 (12 municipalités) → output_energyscope/C4/Demands.csv
  HOUSEHOLDS        :  47.7672 GWh
  SERVICES          :   8.7103 GWh
  INDUSTRY          :   1.5137 GWh
  PUBLIC_LIGHTING   :   0.4019 GWh
  TOTAL             :  58.3931 GWh
----------------------------------------------
Cluster 5 (1 municipalités) → output_energyscope/C5/Demands.csv
  HOUSEHOLDS        :  51.4099 GWh
  SERVICES          :  21.0095 GWh
  INDUSTRY          :   3.3776 GWh
  PUBLIC_LIGHTING   :   0.7523 GWh
  TOTAL             :  76.5492 GWh
----------------------------------------------


## 11. Vérification 2 — `access = reality − B_reality + (B+C)` colonne par colonne

Reconstruit la table attendue à partir des `Demands.csv` **reality existants** et compare
chaque cellule (21 lignes × 8 secteurs) à ce que produit ce notebook. Contrôle indépendant :
il repart des fichiers `reality` sur disque, pas des intermédiaires calculés ici.

In [13]:
TOL = 1e-9
all_ok = True

print("VÉRIFICATION 2 — access = reality − B_reality + (B+C), cellule par cellule")
for cluster_id in sorted(CLUSTERS):
    expected = pd.read_csv(f"../reality/output_energyscope/C{cluster_id}/Demands.csv", sep=";")
    for sec in SECTOR_COLS:
        expected[sec] = expected[sec].astype(float)

    for (sector, end_use), gwh in source_b_reality_by_cluster[cluster_id].items():
        if end_use in MOBILITY_LAYERS or sector not in SECTOR_COLS:
            continue
        expected.loc[expected["parameter name"] == end_use, sector] -= gwh
    add_demands(expected, bc_by_cluster[cluster_id])

    actual = demands_access[cluster_id]
    n_cells, worst = 0, 0.0
    for sec in SECTOR_COLS:
        diff = (actual[sec].astype(float) - expected[sec]).abs()
        n_cells += len(diff)
        worst = max(worst, diff.max())
        for i in diff[diff > TOL].index:
            all_ok = False
            print(f"  ✗ C{cluster_id} {actual.loc[i,'parameter name']:<30} {sec:<16} "
                  f"attendu={expected.loc[i,sec]:.6f} obtenu={float(actual.loc[i,sec]):.6f}")
    print(f"  C{cluster_id}: {n_cells} cellules comparées — écart max {worst:.2e} — "
          f"{'OK' if worst <= TOL else 'ÉCHEC'}")

print()
print("→ TOUTES LES COLONNES CONCORDENT" if all_ok else "→ DES ÉCARTS SUBSISTENT")
assert all_ok

VÉRIFICATION 2 — access = reality − B_reality + (B+C), cellule par cellule
  C1: 168 cellules comparées — écart max 2.22e-16 — OK
  C2: 168 cellules comparées — écart max 8.33e-17 — OK
  C3: 168 cellules comparées — écart max 3.55e-15 — OK


  C4: 168 cellules comparées — écart max 4.44e-16 — OK


  C5: 168 cellules comparées — écart max 5.55e-17 — OK

→ TOUTES LES COLONNES CONCORDENT


## 12. Vérification 3 — somme régionale avant / après

In [14]:
rows = []
for cluster_id in sorted(CLUSTERS):
    dr = pd.read_csv(f"../reality/output_energyscope/C{cluster_id}/Demands.csv", sep=";")
    before = sum(dr[s].astype(float).sum() for s in SECTOR_COLS)
    after  = sum(demands_access[cluster_id][s].astype(float).sum() for s in SECTOR_COLS)
    rows.append({"Cluster": f"C{cluster_id}", "reality_GWh": before, "access_GWh": after,
                 "delta_GWh": after - before, "delta_%": 100 * (after / before - 1)})

check3 = pd.DataFrame(rows)
check3.loc[len(check3)] = {
    "Cluster": "RÉGION",
    "reality_GWh": check3["reality_GWh"].sum(),
    "access_GWh":  check3["access_GWh"].sum(),
    "delta_GWh":   check3["delta_GWh"].sum(),
    "delta_%":     100 * (check3["access_GWh"].sum() / check3["reality_GWh"].sum() - 1),
}

print("VÉRIFICATION 3 — somme régionale avant / après")
print(check3.to_string(index=False, formatters={
    "reality_GWh": "{:9.4f}".format, "access_GWh": "{:9.4f}".format,
    "delta_GWh": "{:+9.4f}".format, "delta_%": "{:+7.2f}".format}))

b_removed = sum(sum(v.values()) for v in source_b_reality_by_cluster.values())
bc_added  = sum(sum(v.values()) for v in bc_by_cluster.values())
print()
print(f"Décomposition du delta régional : −{b_removed:.4f} (B reality retirée) "
      f"+{bc_added:.4f} (B+C sufficiency) = {bc_added - b_removed:+.4f} GWh")
assert abs((bc_added - b_removed) - check3.loc[len(check3)-1, "delta_GWh"]) < 1e-9

VÉRIFICATION 3 — somme régionale avant / après
Cluster reality_GWh access_GWh delta_GWh delta_%
     C1     26.3295    33.7072   +7.3778  +28.02
     C2      1.4126     2.3661   +0.9535  +67.50
     C3    144.3144   153.9374   +9.6230   +6.67
     C4     46.5282    58.3931  +11.8650  +25.50
     C5     75.6237    76.5492   +0.9255   +1.22
 RÉGION    294.2083   324.9531  +30.7448  +10.45

Décomposition du delta régional : −2.1969 (B reality retirée) +32.9417 (B+C sufficiency) = +30.7448 GWh


## 13. Tableau final — demandes par cluster et usage final, avec provenance

In [15]:
PROVENANCE = {
    "A_réseau_reality":   source_a_by_cluster,
    "BC_sufficiency":     bc_by_cluster,
}

rows = []
for cluster_id in sorted(CLUSTERS):
    per_source = {name: d.get(cluster_id, {}) for name, d in PROVENANCE.items()}
    keys = {k for d in per_source.values() for k in d}
    keys |= {("HOUSEHOLDS", "COOKING")}
    for sector, end_use in sorted(keys):
        if end_use in MOBILITY_LAYERS or sector not in SECTOR_COLS:
            continue
        a  = per_source["A_réseau_reality"].get((sector, end_use), 0.0)
        bc = per_source["BC_sufficiency"].get((sector, end_use), 0.0)
        ck = cooking_by_cluster[cluster_id] if (sector, end_use) == ("HOUSEHOLDS", "COOKING") else 0.0
        rows.append({"Cluster": f"C{cluster_id}", "Secteur": sector, "Usage_final": end_use,
                     "A_réseau_reality": a, "Cooking_reality": ck, "BC_sufficiency": bc,
                     "TOTAL_access": a + ck + bc})

final = pd.DataFrame(rows).sort_values(["Cluster", "Secteur", "Usage_final"]).reset_index(drop=True)
final.to_csv(f"{OUTPUT_DIR}/demands_access_by_provenance.csv", index=False)

print("Demandes reality_access par cluster et usage final — GWh utile/an")
print(final.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
print()
print("Totaux par provenance (GWh utile/an) :")
print(final[["A_réseau_reality", "Cooking_reality", "BC_sufficiency", "TOTAL_access"]]
      .sum().to_string(float_format=lambda v: f"{v:.4f}"))
print(f"\n→ {OUTPUT_DIR}/demands_access_by_provenance.csv")

Demandes reality_access par cluster et usage final — GWh utile/an
Cluster         Secteur            Usage_final  A_réseau_reality  Cooking_reality  BC_sufficiency  TOTAL_access
     C1      HOUSEHOLDS                COOKING            0.0000          14.3784          0.0000       14.3784
     C1      HOUSEHOLDS            ELECTRICITY            1.0420           0.0000          0.3592        1.4012
     C1      HOUSEHOLDS      FOOD_PRESERVATION            1.0957           0.0000          1.6571        2.7528
     C1      HOUSEHOLDS          HEAT_LOW_T_HW            0.3120           0.0000          4.3310        4.6429
     C1      HOUSEHOLDS           LIGHTING_R_C            0.3559           0.0000          0.0968        0.4527
     C1      HOUSEHOLDS          SPACE_COOLING            4.7380           0.0000          1.0071        5.7452
     C1        INDUSTRY           LIGHTING_R_C            0.0128           0.0000          0.0000        0.0128
     C1        INDUSTRY  MECHANICAL_EN